# 00b. Vector Data Consolidation

Google Drive 내 산발적으로 존재하는 벡터 파일들을 `Final_Vector/` 폴더로 통합.

**입력 구조** (현재 상태):
```
URP/
  1224_Vectors/
    Inter_PI/    Inter_PI_1.npz ~ Inter_PI_512.npz
    3D_PI/       3D_PI_1.npz ~ 3D_PI_512.npz
    Ord_PI/      Ord_PI_1.npz ~ Ord_PI_512.npz
    Sixpack_Rips/    Sixpack_Rips_1.npz ~ Sixpack_Rips_512.npz
    Sixpack_Chroma/  Sixpack_Chroma_1.npz ~ Sixpack_Chroma_512.npz
```

**출력 구조** (통합 후):
```
URP/Final_Vector/
  Inter_PI/    (복사)
  3D_PI/       (복사)
  Ord_PI/      (복사)
  Sixpack_Rips/    (복사)
  Sixpack_Chroma/  (복사)
  manifest.csv     (파일 목록 & 무결성 정보)
```

## 1. 환경 설정

In [ ]:
import os, glob, shutil, time
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/URP'
TOTAL_SIMS = 512  # 8×8×8 파라미터 조합

# ============================================================
# 소스 경로 설정
# 벡터 파일이 여러 곳에 흩어져 있다면 아래에 추가 경로를 넣으세요.
# ============================================================
SOURCE_DIRS = {
    'Inter_PI': [
        os.path.join(BASE_DIR, '1224_Vectors', 'Inter_PI'),
        # os.path.join(BASE_DIR, 'extra_vectors', 'Inter_PI'),  # 추가 경로 예시
    ],
    '3D_PI': [
        os.path.join(BASE_DIR, '1224_Vectors', '3D_PI'),
    ],
    'Ord_PI': [
        os.path.join(BASE_DIR, '1224_Vectors', 'Ord_PI'),
    ],
    'Sixpack_Rips': [
        os.path.join(BASE_DIR, '1224_Vectors', 'Sixpack_Rips'),
    ],
    'Sixpack_Chroma': [
        os.path.join(BASE_DIR, '1224_Vectors', 'Sixpack_Chroma'),
    ],
}

# 출력 경로
OUTPUT_BASE = os.path.join(BASE_DIR, 'Final_Vector')
os.makedirs(OUTPUT_BASE, exist_ok=True)

print(f'Source base: {BASE_DIR}')
print(f'Output base: {OUTPUT_BASE}')

## 2. 소스 스캔 & 현황 파악

In [ ]:
def scan_sources(source_dirs, descriptor):
    """여러 소스 폴더에서 해당 descriptor의 .npz 파일을 수집.
    같은 sim_idx가 여러 곳에 있으면 가장 최근 파일 사용."""
    files = {}  # sim_idx → filepath
    for src_dir in source_dirs:
        if not os.path.exists(src_dir):
            print(f'  [SKIP] {src_dir} (not found)')
            continue
        for fp in glob.glob(os.path.join(src_dir, f'{descriptor}_*.npz')):
            try:
                sim_idx = int(os.path.basename(fp).split('_')[-1].split('.')[0])
                if sim_idx not in files or os.path.getmtime(fp) > os.path.getmtime(files[sim_idx]):
                    files[sim_idx] = fp
            except ValueError:
                pass
    return files

print('=' * 70)
print('소스 스캔 결과')
print('=' * 70)

all_scanned = {}
for desc, dirs in SOURCE_DIRS.items():
    files = scan_sources(dirs, desc)
    all_scanned[desc] = files
    found = set(files.keys())
    missing = set(range(1, TOTAL_SIMS+1)) - found
    status = '✓ COMPLETE' if len(missing) == 0 else f'✗ MISSING {len(missing)}'
    print(f'  {desc:<20s} : {len(found):>4d}/{TOTAL_SIMS}  {status}')
    if missing and len(missing) <= 20:
        print(f'    Missing IDs: {sorted(missing)}')
    elif missing:
        print(f'    Missing IDs (first 20): {sorted(missing)[:20]} ...')

## 3. 파일 복사 & 통합

In [ ]:
manifest_rows = []

for desc in SOURCE_DIRS:
    dest_dir = os.path.join(OUTPUT_BASE, desc)
    os.makedirs(dest_dir, exist_ok=True)
    files = all_scanned[desc]
    copied, skipped = 0, 0

    for sim_idx in sorted(files.keys()):
        src_path = files[sim_idx]
        dest_filename = f'{desc}_{sim_idx}.npz'
        dest_path = os.path.join(dest_dir, dest_filename)

        # 이미 존재하면 크기 비교 후 스킵
        if os.path.exists(dest_path):
            if os.path.getsize(dest_path) == os.path.getsize(src_path):
                skipped += 1
                manifest_rows.append({
                    'descriptor': desc, 'sim_idx': sim_idx,
                    'filename': dest_filename, 'size_bytes': os.path.getsize(dest_path),
                    'status': 'exists'
                })
                continue

        shutil.copy2(src_path, dest_path)
        copied += 1
        manifest_rows.append({
            'descriptor': desc, 'sim_idx': sim_idx,
            'filename': dest_filename, 'size_bytes': os.path.getsize(dest_path),
            'status': 'copied'
        })

    print(f'{desc:<20s}: copied={copied}, skipped(exists)={skipped}, total={len(files)}')

print(f'\n총 {len(manifest_rows)}개 파일 처리 완료')

## 4. Manifest 저장 & 무결성 검증

In [ ]:
# Manifest CSV 저장
manifest_df = pd.DataFrame(manifest_rows)
manifest_path = os.path.join(OUTPUT_BASE, 'manifest.csv')
manifest_df.to_csv(manifest_path, index=False)
print(f'Manifest 저장: {manifest_path}')
print(f'총 {len(manifest_df)}개 레코드\n')

# 무결성 검증: 각 descriptor 별 파일 로드 테스트
print('=' * 70)
print('무결성 검증 (샘플 로드 테스트)')
print('=' * 70)

for desc in SOURCE_DIRS:
    dest_dir = os.path.join(OUTPUT_BASE, desc)
    files = sorted(glob.glob(os.path.join(dest_dir, f'{desc}_*.npz')))
    if not files:
        print(f'  {desc}: [EMPTY]')
        continue

    # 첫 파일, 중간 파일, 마지막 파일 테스트
    test_indices = [0, len(files)//2, len(files)-1]
    ok_count, err_count = 0, 0
    for ti in test_indices:
        try:
            data = np.load(files[ti], allow_pickle=True)
            keys = list(data.keys())
            ok_count += 1
        except Exception as e:
            err_count += 1
            print(f'  {desc}: ERROR loading {files[ti]} - {e}')

    print(f'  {desc:<20s}: {len(files):>4d} files, sample test {ok_count}/{ok_count+err_count} OK')

## 5. 최종 요약

In [ ]:
print('=' * 70)
print('Final_Vector 최종 구조')
print('=' * 70)

total_size = 0
for desc in SOURCE_DIRS:
    dest_dir = os.path.join(OUTPUT_BASE, desc)
    files = glob.glob(os.path.join(dest_dir, '*.npz'))
    size = sum(os.path.getsize(f) for f in files)
    total_size += size
    missing = TOTAL_SIMS - len(files)
    status = '✓' if missing == 0 else f'✗ ({missing} missing)'
    print(f'  {dest_dir}')
    print(f'    {len(files):>4d} files, {size/1024/1024:.1f} MB  {status}')

print(f'\n  Total: {total_size/1024/1024:.1f} MB')
print(f'  Manifest: {manifest_path}')
print(f'\n이제 01_Setup_and_Data_Loading.ipynb에서')
print(f"VECTOR_DIR = '{OUTPUT_BASE}'")
print('로 경로를 변경하면 통합 데이터를 사용할 수 있습니다.')

## (선택) 추가 소스 검색
특정 폴더 이름 패턴으로 Drive 전체를 검색하여 누락된 벡터 파일을 찾습니다.

In [ ]:
# Drive 전체에서 특정 descriptor 파일 검색 (시간이 오래 걸릴 수 있음)
# 필요할 때만 실행하세요.

SEARCH_ROOT = '/content/drive/MyDrive'  # 검색 시작 경로
SEARCH_DESC = 'Sixpack_Rips'  # 검색할 descriptor

def search_drive(root, descriptor, max_depth=4):
    found = {}
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.replace(root, '').count(os.sep)
        if depth > max_depth:
            dirnames.clear()
            continue
        for fn in filenames:
            if fn.startswith(f'{descriptor}_') and fn.endswith('.npz'):
                try:
                    sim_idx = int(fn.split('_')[-1].split('.')[0])
                    fp = os.path.join(dirpath, fn)
                    if sim_idx not in found:
                        found[sim_idx] = fp
                except ValueError:
                    pass
    return found

# print(f'Searching for {SEARCH_DESC} in {SEARCH_ROOT}...')
# extra = search_drive(SEARCH_ROOT, SEARCH_DESC)
# print(f'Found {len(extra)} files')
# for idx in sorted(extra)[:10]:
#     print(f'  {idx}: {extra[idx]}')